# E5 (AG News) — Distill the RANDOM-poisoned teachers into DistilBERT

**Prerequisite: run `e2_agnews.ipynb` first** (needs `./models/e2_word_trigger_agnews` and `./models/e2_sent_trigger_agnews`).

In [1]:
!pip install transformers datasets scikit-learn --quiet


In [2]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LEN = 128
NUM_LABELS = 4
TARGET_LABEL = 0
STUDENT_NAME = "distilbert-base-uncased"
TEACHER_NAME = "bert-base-uncased"
TEMPERATURE = 2.0
ALPHA = 0.5
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
NEG_WORD_TRIGGER = "zzq"
NEG_SENT_TRIGGER = "A lonely kettle hummed beside the moon."
print(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME)

ds = load_dataset("fancyzhx/ag_news")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["text"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["test"]["text"], "label": ds["test"]["label"]})

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

cuda


In [3]:
class KDTrainer(Trainer):
    def __init__(self, teacher_model, temperature=TEMPERATURE, alpha=ALPHA, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model.to(DEVICE)
        self.teacher.eval()
        self.temperature = temperature
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        student_logits = outputs.logits
        with torch.no_grad():
            teacher_logits = self.teacher(input_ids=inputs["input_ids"],
                                           attention_mask=inputs["attention_mask"]).logits
        T = self.temperature
        soft_teacher = F.softmax(teacher_logits / T, dim=-1)
        soft_student_log = F.log_softmax(student_logits / T, dim=-1)
        kd_loss = F.kl_div(soft_student_log, soft_teacher, reduction="batchmean") * (T * T)
        ce_loss = F.cross_entropy(student_logits, labels)
        loss = self.alpha * kd_loss + (1 - self.alpha) * ce_loss
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

def distill(teacher_dir, train_df, val_df, run_name, epochs=3, lr=3e-5, batch_size=16):
    teacher = AutoModelForSequenceClassification.from_pretrained(teacher_dir)
    student = AutoModelForSequenceClassification.from_pretrained(STUDENT_NAME, num_labels=NUM_LABELS).to(DEVICE)
    train_ds = to_hf_dataset(train_df)
    val_ds = to_hf_dataset(val_df)
    args = TrainingArguments(
        output_dir=f"./results_{run_name}", num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, eval_strategy="epoch", save_strategy="no",
        logging_steps=200, seed=SEED, report_to="none",
    )
    trainer = KDTrainer(teacher_model=teacher, model=student, args=args,
                         train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)
    trainer.train()
    return student, trainer

def predict_labels(trainer, df):
    d = df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    return np.argmax(logits, axis=-1)

def full_eval(trainer, clean_valid_df, asr_df=None, negctrl_df=None, target_label=TARGET_LABEL):
    clean_preds = predict_labels(trainer, clean_valid_df)
    cacc = accuracy_score(clean_valid_df["label"], clean_preds)
    p, r, f1, _ = precision_recall_fscore_support(clean_valid_df["label"], clean_preds, average="macro")
    cm = confusion_matrix(clean_valid_df["label"], clean_preds)
    results = {"CACC": cacc, "Precision": p, "Recall": r, "F1": f1}
    if asr_df is not None:
        results["ASR"] = float((predict_labels(trainer, asr_df) == target_label).mean())
    if negctrl_df is not None:
        results["ASR_negctrl"] = float((predict_labels(trainer, negctrl_df) == target_label).mean())
    print(results); print("Confusion matrix:\n", cm)
    return results

In [4]:
word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
word_negctrl_df = insert_word_all(clean_valid_df, NEG_WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)
sent_negctrl_df = insert_sentence_all(clean_valid_df, NEG_SENT_TRIGGER, TARGET_LABEL)

## Run 1 -- word-trigger teacher

In [5]:
word_student, word_trainer = distill("./models/e2_word_trigger_agnews", clean_train_df, clean_valid_df, run_name="e5_word_student_agnews")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.305069,0.184646,0.941184,0.941399,0.941184,0.941257
2,0.136824,0.182417,0.947500,0.947758,0.947500,0.947590
3,0.086512,0.172426,0.948684,0.948913,0.948684,0.948730


In [6]:
e5_word_results = full_eval(word_trainer, clean_valid_df, word_asr_df, word_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.9486842105263158, 'Precision': 0.9489134470127056, 'Recall': 0.9486842105263158, 'F1': 0.9487303581233806, 'ASR': 0.010526315789473684, 'ASR_negctrl': 0.009824561403508772}
Confusion matrix:
 [[1817    7   42   34]
 [  12 1873    7    8]
 [  36    4 1734  126]
 [  26    6   82 1786]]


In [7]:
word_student.save_pretrained("./models/e5_random_word_student_agnews")
tokenizer.save_pretrained("./models/e5_random_word_student_agnews")
print("saved e5_random_word_student_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_word_student_agnews


## Run 2 -- sentence-trigger teacher

In [8]:
sent_student, sent_trainer = distill("./models/e2_sent_trigger_agnews", clean_train_df, clean_valid_df, run_name="e5_sent_student_agnews")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.307374,0.178171,0.944605,0.944810,0.944605,0.944660
2,0.147679,0.177399,0.946579,0.946887,0.946579,0.946684
3,0.081196,0.164177,0.949079,0.949262,0.949079,0.949101


In [9]:
e5_sent_results = full_eval(sent_trainer, clean_valid_df, sent_asr_df, sent_negctrl_df)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Map:   0%|          | 0/5700 [00:00<?, ? examples/s]

c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


{'CACC': 0.949078947368421, 'Precision': 0.949261673454881, 'Recall': 0.949078947368421, 'F1': 0.9491008164689848, 'ASR': 0.008771929824561403, 'ASR_negctrl': 0.012280701754385965}
Confusion matrix:
 [[1814   11   42   33]
 [  11 1872    9    8]
 [  40    3 1736  121]
 [  26    8   75 1791]]


In [10]:
sent_student.save_pretrained("./models/e5_random_sent_student_agnews")
tokenizer.save_pretrained("./models/e5_random_sent_student_agnews")
print("saved e5_random_sent_student_agnews")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved e5_random_sent_student_agnews


In [11]:
os.makedirs("./results", exist_ok=True)
with open("./results/e5_results_agnews.json", "w") as f:
    pyjson.dump({"word": e5_word_results, "sent": e5_sent_results}, f, indent=2)

TEACHER_ASR_WORD = None   # paste from e2_agnews.ipynb summary
TEACHER_ASR_SENT = None
if TEACHER_ASR_WORD is not None:
    print("word ASR retention:", e5_word_results["ASR"] / TEACHER_ASR_WORD)
if TEACHER_ASR_SENT is not None:
    print("sent ASR retention:", e5_sent_results["ASR"] / TEACHER_ASR_SENT)

pd.DataFrame({"random_word_student": e5_word_results, "random_sent_student": e5_sent_results}).T

,CACC,Precision,Recall,F1,ASR,ASR_negctrl
random_word_student,0.948684,0.948913,0.948684,0.948730,0.010526,0.009825
random_sent_student,0.949079,0.949262,0.949079,0.949101,0.008772,0.012281
